In [94]:
import os
import geopandas as gpd
import pandas as pd

#Get current working directory
project_folder = os.getcwd()

#Build full path to the GeoPackage
farm_file_path = "C:/Users/Vasmai/dataset/farm_boundaries_sampled.gpkg"

#Load the spatial farm boundary dataset
farm_boundaries = gpd.read_file(farm_file_path)

#get first few records
print(farm_boundaries.head())

#Display column names
print(farm_boundaries.columns)

#Export the full dataset to CSV
csv_file_path = os.path.join("C:/Users/Vasmai/dataset", "farm_boundaries.csv")

farm_boundaries['geometry'] = farm_boundaries['geometry'].apply(lambda x: x.wkt)
farm_boundaries.to_csv(csv_file_path, index=False)

print(f"\nGeoPackage exported to CSV: {csv_file_path}")
# Check dataset structure, data types, and missing values
farm_boundaries.info()

# Dataset not included in GitHub due to privacy
# Can be downloaded from MS Teams → Planting Optimisation Tool → Datasets → GIS

                name treeo2_id  district subdistrict         suku  \
0     Abel Pereira_1              BAUCAU      BAGUIA     Lavateri   
1     Abel Pereira_2              BAUCAU      BAGUIA     Lavateri   
2    Agostinho Alves            VIQUEQUE  UATUCARBAU  Alaua-Craik   
3  Agostinho Freitas            VIQUEQUE  UATUCARBAU  Alaua-Craik   
4     Amelia Menezes              BAUCAU      BAGUIA  Alaua-Craik   

         aldeia     texture   ph  avg_annual_rainfall_2020-2024  \
0  Onortibalari        Clay  6.2                           1959   
1  Onortibalari        Clay  6.2                           1959   
2      Neolidae  Sandy Loam  8.2                           2021   
3      Neolidae  Sandy Loam  5.9                           2554   
4      Neolidae        Clay  7.0                           2021   

   avg_annual_temperature_2020-2024  elevation          area  area_ha  \
0                              24.0        585   3596.997671     0.36   
1                              24.0 

C:\Users\Vasmai\AppData\Local\Temp\ipykernel_22572\2589199513.py:23: UserWarning: Geometry column does not contain geometry.
  farm_boundaries['geometry'] = farm_boundaries['geometry'].apply(lambda x: x.wkt)


#### Remove Privacy Information

In [83]:
#getrid of privacy info
cols_to_remove = ['name', 'suku', 'aldeia', 'treeo2_id', 'district', 'subdistrict', 'area']
farm_cleaned = farm_boundaries.drop(columns=cols_to_remove, errors='ignore')

# Preview cleaned dataset
print("privacy info removed:")

# Check cleaned dataset info
farm_cleaned.info()

farm_cleaned.head()

privacy info removed:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 3201 entries, 0 to 3200
Data columns (total 7 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   texture                           3154 non-null   object  
 1   ph                                3154 non-null   object  
 2   avg_annual_rainfall_2020-2024     3201 non-null   int32   
 3   avg_annual_temperature_2020-2024  3193 non-null   float64 
 4   elevation                         3201 non-null   int32   
 5   area_ha                           3201 non-null   float64 
 6   geometry                          3201 non-null   geometry
dtypes: float64(2), geometry(1), int32(2), object(2)
memory usage: 150.2+ KB


,texture,ph,avg_annual_rainfall_2020-2024,avg_annual_temperature_2020-2024,elevation,area_ha,geometry
0,Clay,6.2,1959,24.0,585,0.36,"MULTIPOLYGON Z (((904783.597 9050919.352 0, 90..."
1,Clay,6.2,1959,24.0,481,0.48,"MULTIPOLYGON Z (((905235.474 9051056.651 0, 90..."
2,Sandy Loam,8.2,2021,25.0,179,1.18,"MULTIPOLYGON Z (((903414.205 9042747.721 0, 90..."
3,Sandy Loam,5.9,2554,24.0,260,0.46,"MULTIPOLYGON Z (((902007.836 9043097.668 0, 90..."
4,Clay,7.0,2021,25.0,129,2.00,"MULTIPOLYGON Z (((904016.568 9042595.439 0, 90..."


#### Rename Columns

In [84]:
# add farm_id and rename columns
farm_cleaned.insert(0, 'farm_id', range(1, len(farm_boundaries) + 1))
farm_cleaned = farm_cleaned.rename(columns={'avg_annual_rainfall_2020-2024': 'rainfall','avg_annual_temperature_2020-2024': 'temperature'})
farm_cleaned.info()
farm_cleaned.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 3201 entries, 0 to 3200
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   farm_id      3201 non-null   int64   
 1   texture      3154 non-null   object  
 2   ph           3154 non-null   object  
 3   rainfall     3201 non-null   int32   
 4   temperature  3193 non-null   float64 
 5   elevation    3201 non-null   int32   
 6   area_ha      3201 non-null   float64 
 7   geometry     3201 non-null   geometry
dtypes: float64(2), geometry(1), int32(2), int64(1), object(2)
memory usage: 175.2+ KB


,farm_id,texture,ph,rainfall,temperature,elevation,area_ha,geometry
0,1,Clay,6.2,1959,24.0,585,0.36,"MULTIPOLYGON Z (((904783.597 9050919.352 0, 90..."
1,2,Clay,6.2,1959,24.0,481,0.48,"MULTIPOLYGON Z (((905235.474 9051056.651 0, 90..."
2,3,Sandy Loam,8.2,2021,25.0,179,1.18,"MULTIPOLYGON Z (((903414.205 9042747.721 0, 90..."
3,4,Sandy Loam,5.9,2554,24.0,260,0.46,"MULTIPOLYGON Z (((902007.836 9043097.668 0, 90..."
4,5,Clay,7.0,2021,25.0,129,2.00,"MULTIPOLYGON Z (((904016.568 9042595.439 0, 90..."


#### Missing Values

In [90]:
farm_cleaned.isnull().sum()

farm_id         0
texture        47
ph             47
rainfall        0
temperature     8
elevation       0
area_ha         0
geometry        0
dtype: int64

In [100]:
# Fill missing pH values
farm_cleaned['ph'] = farm_cleaned['ph'].fillna(farm_cleaned['ph'].median())

# Fill missing temperature values
farm_cleaned['temperature'] = farm_cleaned['temperature'].fillna(farm_cleaned['temperature'].median())

# Fill missing texture values
farm_cleaned['texture'] = farm_cleaned['texture'].fillna(farm_cleaned['texture'].mode()[0])

# Check missing values again
print(farm_cleaned.isnull().sum())
print(farm_cleaned.head())

farm_id        0
texture        0
ph             0
rainfall       0
temperature    0
elevation      0
area_ha        0
geometry       0
dtype: int64
   farm_id     texture   ph  rainfall  temperature  elevation  area_ha  \
0        1        Clay  6.2      1959         24.0        585     0.36   
1        2        Clay  6.2      1959         24.0        481     0.48   
2        3  Sandy Loam  8.2      2021         25.0        179     1.18   
3        4  Sandy Loam  5.9      2554         24.0        260     0.46   
4        5        Clay  7.0      2021         25.0        129     2.00   

                                            geometry  
0  MULTIPOLYGON Z (((904783.5966190484 9050919.35...  
1  MULTIPOLYGON Z (((905235.4744740495 9051056.65...  
2  MULTIPOLYGON Z (((903414.2049201063 9042747.72...  
3  MULTIPOLYGON Z (((902007.836062541 9043097.667...  
4  MULTIPOLYGON Z (((904016.5677603828 9042595.43...  


In [98]:
#Export the full dataset to CSV
csv_file_path = os.path.join("C:/Users/Vasmai/dataset", "farm_boundaries_cleaned.csv")

farm_cleaned['geometry'] = farm_cleaned['geometry'].apply(lambda x: x.wkt)
farm_cleaned.to_csv(csv_file_path, index=False)

print(f"\nGeoPackage exported to CSV: {csv_file_path}")

C:\Users\Vasmai\AppData\Local\Temp\ipykernel_22572\2295350178.py:4: UserWarning: Geometry column does not contain geometry.
  farm_cleaned['geometry'] = farm_cleaned['geometry'].apply(lambda x: x.wkt)



GeoPackage exported to CSV: C:/Users/Vasmai/dataset\farm_boundaries_cleaned.csv
